<a href="https://colab.research.google.com/github/kili-technology/kili-python-sdk/blob/main/recipes/importing_multilayer_geospatial_assets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Multi-Layer Geospatial Images into a Kili Project

In this tutorial, we will learn how to import multi-layer geospatial assets to your project.

Here are the steps that we will follow:

1. Setting up an image project.
2. Importing some multi-layer assets to Kili with GEOTIFFs and layers from a public tile server.

## Setting up a Kili image project to work with

### Installing and instantiating Kili

First, let's install and import the required modules.

In [ ]:
%pip install kili

In [ ]:
from kili.client_domain import Kili

Now, let's set up variables needed to create an instance of the Kili object.

We will need your API key and Kili's API endpoint.

If you are unsure how to look up your API key, refer to [https://docs.kili-technology.com/docs/creating-an-api-key](https://docs.kili-technology.com/docs/creating-an-api-key).

In [ ]:
api_key = "YOUR_API_KEY"
api_endpoint = "https://cloud.kili-technology.com/api/label/v2/graphql"
kili = Kili(
    # api_endpoint=api_endpoint, api_key=api_key, verify=True
    # the line above can be uncommented and changed if you are working with an on-premise version of Kili
)

### Creating an image Kili project

To create an IMAGE Kili project, you must first set up its ontology.

Here, we will only add a classification task:

In [ ]:
interface = {
    "jobs": {
        "OBJECT_DETECTION_JOB": {
            "content": {
                "categories": {
                    "PLANE": {
                        "children": [],
                        "color": "#472CED",
                        "name": "Plane",
                        "id": "category3",
                    }
                },
                "input": "radio",
            },
            "instruction": "BBOX",
            "mlTask": "OBJECT_DETECTION",
            "required": 0,
            "tools": ["rectangle"],
            "isChild": False,
            "isNew": False,
        }
    }
}

project = kili.projects.create(
    title="[Kili SDK Notebook]: Importing multi-layer Geospatial asset",
    description="Project Description",
    input_type="IMAGE",
    json_interface=interface,
)

## Importing your assets

### Download geotiff file examples

Before adding assets you need to download our geotiff examples and add them in a geospatial folder, created in the same folder as where you run your python script. It is mandatory to use local files and not urls for geospatial files.

In [ ]:
import os
import urllib.request

if not os.path.exists("geospatial"):
    os.makedirs("geospatial")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/label-public-staging/asset-test-sample/geospatial/a.tiff",
    "geospatial/a.tiff",
)
urllib.request.urlretrieve(
    "https://storage.googleapis.com/label-public-staging/asset-test-sample/geospatial/b.tiff",
    "geospatial/b.tiff",
)

### Add Multi-Layer assets to your project

You can now add assets. Here is an example to add 2 geotiffs and a public layer coming from the openstreetmap tile server.

In [ ]:
project_id = project["id"]
kili.assets.create_geospatial(
    project_id=project_id,
    layer_array=[
        {"path": "geospatial/a.tiff", "name": "Layer 1", "is_base_layer": False},
        {"path": "geospatial/b.tiff", "name": "Layer 2", "is_base_layer": False},
        {
            "path": "https://b.tile.openstreetmap.org/{z}/{x}/{y}.png",
            "name": "osm",
            "epsg": "EPSG3857",
            "bounds": [
                [11.17010498046875, 44.308941579503745],
                [13.67478942871094, 46.542667432984864],
            ],
            "min_zoom": 10,
            "max_zoom": 18,
            "is_base_layer": True,
        },
    ],
    json_metadata={"processingParameters": {"epsg": 3857}},
)

In this example, 3 arguments are given to `kili.assets.create_geospatial`:

1. `project_id`: the id of the project to which you want to add the asset
1. `layer_array`: the layers of the asset. A layer whose `path` is a local file, like a GEOTIFF, is uploaded and tiled by Kili; a layer whose `path` is a URL is a public tile layer. Each layer takes the `name` that will be used in kili for the layer and the boolean `is_base_layer` to define if it's a base layer (only one visible at a time) or an overlay layer (a layer that will be displayed on top of the base layer). It is optional: by default, a GEOTIFF is a base layer and a tile layer an overlay. A tile layer also takes:
    1. `epsg`, with the format `EPSG{number}`: the EPSG used by the tile server, as we will not reproject anything;
    1. `bounds`, the minimum and maximum latitude and longitude for which you want to request tiles, with the format `[[min_lng, min_lat],[max_lng, max_lat]]`;
    1. `min_zoom` and `max_zoom`, the zoom levels between which tiles are requested.
1. `json_metadata`: the processing parameters used when processing the GEOTIFFS. 3 parameters can be set there :
    1. `epsg`: This one defines the projection (<https://en.wikipedia.org/wiki/EPSG_Geodetic_Parameter_Dataset>) to which we will reproject the dataset. Our frontend supports only two projections : `EPSG:4326` and `EPSG:3857`. If this parameter is not set we will keep the projection of the initial file if it is one of these two, else we will reproject it by default to `EPSG:3857`. By default, we advise to not set this parameter but if you need to use your GEOTIFFS with some tile server (as with openstreetmap here) you will need to reproject it to the same EPSG as the one used by the tile server as our application supports only one EPSG for the whole asset. For your information most of the well known tile server (openstreetmap, googlemaps, etc) are using `EPSG:3857`.
    1. `maxZoom` and `minZoom`: these defines limits of zoom for your GEOTIFF files. This is especially useful for files that will be tiled by our server (file size > 30MB). By default we generate all the zooms until the one of the original file but if you want to limit to specific zoom levels you can constrain them with these parameters.

## Cleanup

We can remove the project that we created if needed:

In [ ]:
kili.projects.delete(project_id=project_id)